In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkConf

conf = SparkConf()
conf.set('spark.app.name', 'PySpark File Format')
conf.set('spark.master', 'local[*]')

spark = SparkSession.builder\
        .config(conf = conf)\
        .getOrCreate()

## 파티션 변경 및 여러 파일 포맷들로 저장하기

In [6]:
df = spark.read.csv('appl_stock.csv', header = True)

df.show(5)

+----------+----------+----------+------------------+------------------+---------+------------------+
|      Date|      Open|      High|               Low|             Close|   Volume|         Adj Close|
+----------+----------+----------+------------------+------------------+---------+------------------+
|2010-01-04|213.429998|214.499996|212.38000099999996|        214.009998|123432400|         27.727039|
|2010-01-05|214.599998|215.589994|        213.249994|        214.379993|150476200|27.774976000000002|
|2010-01-06|214.379993|    215.23|        210.750004|        210.969995|138040000|27.333178000000004|
|2010-01-07|    211.75|212.000006|        209.050005|            210.58|119282800|          27.28265|
|2010-01-08|210.299994|212.000006|209.06000500000002|211.98000499999998|111902700|         27.464034|
+----------+----------+----------+------------------+------------------+---------+------------------+
only showing top 5 rows


In [8]:
df.rdd.getNumPartitions()

1

In [10]:
import pyspark.sql.functions as f

df.groupBy(f.spark_partition_id()).count().show()

+--------------------+-----+
|SPARK_PARTITION_ID()|count|
+--------------------+-----+
|                   0| 1762|
+--------------------+-----+



In [12]:
df2 = df.repartition(4)

print(df2.rdd.getNumPartitions())
df2.groupBy(f.spark_partition_id()).count().show()

4
+--------------------+-----+
|SPARK_PARTITION_ID()|count|
+--------------------+-----+
|                   0|  440|
|                   1|  441|
|                   2|  441|
|                   3|  440|
+--------------------+-----+



In [13]:
df3 = df2.coalesce(2)

print(df3.rdd.getNumPartitions())
df3.groupBy(f.spark_partition_id()).count().show()

2
+--------------------+-----+
|SPARK_PARTITION_ID()|count|
+--------------------+-----+
|                   0|  881|
|                   1|  881|
+--------------------+-----+



In [16]:
df2.write.format('parquet')\
    .mode('overwrite')\
    .option('path', 'SaveData/PARQUET')\
    .save()

In [18]:
df3.write.format('json')\
    .mode('overwrite')\
    .option('path', 'SaveData/JSON')\
    .save()

## Schema Evolution

In [3]:
df1 = spark.read.parquet('schema1.parquet')

df1.printSchema()
df1.show(5)

root
 |-- Date: timestamp (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)

+-------------------+----------+----------+------------------+------------------+
|               Date|      Open|      High|               Low|             Close|
+-------------------+----------+----------+------------------+------------------+
|2010-01-04 17:00:00|213.429998|214.499996|212.38000099999996|        214.009998|
|2010-01-05 17:00:00|214.599998|215.589994|        213.249994|        214.379993|
|2010-01-06 17:00:00|214.379993|    215.23|        210.750004|        210.969995|
|2010-01-07 17:00:00|    211.75|212.000006|        209.050005|            210.58|
|2010-01-08 17:00:00|210.299994|212.000006|209.06000500000002|211.98000499999998|
+-------------------+----------+----------+------------------+------------------+
only showing top 5 rows


In [4]:
df2 = spark.read.parquet('schema2.parquet')

df2.printSchema()
df2.show(5)

root
 |-- Date: timestamp (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: integer (nullable = true)

+-------------------+----------+----------+------------------+------------------+---------+
|               Date|      Open|      High|               Low|             Close|   Volume|
+-------------------+----------+----------+------------------+------------------+---------+
|2010-01-04 17:00:00|213.429998|214.499996|212.38000099999996|        214.009998|123432400|
|2010-01-05 17:00:00|214.599998|215.589994|        213.249994|        214.379993|150476200|
|2010-01-06 17:00:00|214.379993|    215.23|        210.750004|        210.969995|138040000|
|2010-01-07 17:00:00|    211.75|212.000006|        209.050005|            210.58|119282800|
|2010-01-08 17:00:00|210.299994|212.000006|209.06000500000002|211.98000499999998|111902700|
+-------------------+----------+----

In [5]:
df3 = spark.read.parquet('schema3.parquet')

df3.printSchema()
df3.show(5)

root
 |-- Date: timestamp (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: integer (nullable = true)
 |-- Adj Close: double (nullable = true)

+-------------------+----------+----------+------------------+------------------+---------+------------------+
|               Date|      Open|      High|               Low|             Close|   Volume|         Adj Close|
+-------------------+----------+----------+------------------+------------------+---------+------------------+
|2010-01-04 17:00:00|213.429998|214.499996|212.38000099999996|        214.009998|123432400|         27.727039|
|2010-01-05 17:00:00|214.599998|215.589994|        213.249994|        214.379993|150476200|27.774976000000002|
|2010-01-06 17:00:00|214.379993|    215.23|        210.750004|        210.969995|138040000|27.333178000000004|
|2010-01-07 17:00:00|    211.75|212.000006|        209.050005|   

In [6]:
df_all = spark.read\
        .option('mergeSchema', 'true')\
        .parquet('*.parquet')

df_all.printSchema()
df_all.show(5)

root
 |-- Date: timestamp (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: integer (nullable = true)
 |-- Adj Close: double (nullable = true)

+-------------------+----------+----------+------------------+------------------+---------+------------------+
|               Date|      Open|      High|               Low|             Close|   Volume|         Adj Close|
+-------------------+----------+----------+------------------+------------------+---------+------------------+
|2010-01-04 17:00:00|213.429998|214.499996|212.38000099999996|        214.009998|123432400|         27.727039|
|2010-01-05 17:00:00|214.599998|215.589994|        213.249994|        214.379993|150476200|27.774976000000002|
|2010-01-06 17:00:00|214.379993|    215.23|        210.750004|        210.969995|138040000|27.333178000000004|
|2010-01-07 17:00:00|    211.75|212.000006|        209.050005|   